# INTELIGÊNCIA ARTIFICIAL
## LABORATÓRIO # 06 — Embeddings: Do Texto ao Vetor
**sentence-transformers · Similaridade de Cosseno · Mini Busca Semântica**

---

### Continuação do Lab 05
No Laboratório 05, você explorou o processamento de texto clássico (tokenização, stopwords, lematização).
Neste laboratório, damos o salto para o processamento de linguagem moderno: transformar frases completas em **vetores numéricos densos (embeddings)** que capturam relações semânticas e significado conceitual.

### Objetivos
1. Gerar representações vetoriais com modelos locais de **Transformers** (`sentence-transformers`).
2. Medir a proximidade semântica entre sentenças através do cálculo da **Similaridade de Cosseno**.
3. Construir uma **Mini Busca Semântica** operando sobre chunks do PDF `introducaoml.pdf`.
4. Comparar arquiteturas locais com embeddings fornecidos via **API (OpenAI)**.


### 1. Instalação e Preparação do Ambiente
Para rodar este laboratório com o modelo local de embeddings, execute:
```bash
pip install sentence-transformers scikit-learn numpy openai
```


In [ ]:
import os
import re
import numpy as np
from sklearn.metrics.pairwise import cosine_similarity

# Carregamento do modelo de embeddings local
try:
    from sentence_transformers import SentenceTransformer
    modelo = SentenceTransformer('paraphrase-multilingual-MiniLM-L12-v2')
    print("Modelo local SentenceTransformer carregado com sucesso!")
except Exception as e:
    print(f"Aviso: Carregando fallback para execução sem GPU/PyTorch: {e}")
    # Classe simuladora para permitir execução didática caso o PyTorch não esteja presente
    class FakeEmbeddingModel:
        def encode(self, texts):
            if isinstance(texts, str): texts = [texts]
            np.random.seed(42)
            # Simula vetores de 384 dimensões mantendo coerência semântica demonstrativa
            res = []
            for t in texts:
                t_lower = t.lower()
                base = np.zeros(384)
                if 'carro' in t_lower or 'veículo' in t_lower or 'defeito' in t_lower or 'quebrou' in t_lower:
                    base[0:10] = [0.8, -0.4, 0.6, 0.3, -0.2, 0.7, 0.5, -0.1, 0.9, -0.3]
                elif 'pizza' in t_lower or 'comida' in t_lower:
                    base[0:10] = [-0.6, 0.7, -0.5, -0.8, 0.4, -0.3, 0.1, 0.8, -0.4, 0.5]
                elif 'inteligência' in t_lower or 'aprendizado' in t_lower or 'máquina' in t_lower or 'dados' in t_lower:
                    base[0:10] = [0.5, 0.9, 0.7, 0.6, -0.5, 0.8, -0.2, 0.4, 0.7, -0.1]
                else:
                    base[0:10] = np.random.uniform(-0.5, 0.5, 10)
                norm = np.linalg.norm(base)
                res.append(base / (norm if norm > 0 else 1.0))
            return np.array(res) if len(texts) > 1 else np.array(res[0])
    modelo = FakeEmbeddingModel()

---
### 2. Gerando o Primeiro Embedding (Do Texto ao Vetor)
Transformamos uma sentença em um vetor denso de 384 dimensões reais.

In [ ]:
frase = "Inteligência artificial é fascinante"
vetor = modelo.encode(frase)

print(f"Texto de entrada: '{frase}'")
print(f"Dimensões do vetor gerado: {vetor.shape}")
print(f"Primeiros 5 componentes numéricos: {np.round(vetor[:5], 4)}")

**Observação Importante:**
A tokenização ocorre internamente dentro de `modelo.encode()`. Ao contrário do PLN clássico do Lab 05, **não fazemos stemming nem remoção de stopwords**, pois o modelo precisa do contexto gramatical completo.


---
### 3. Calculando Similaridade Semântica (Cosseno)
Testamos três frases: duas com vocabulários completamente distintos, mas com mesmo significado prático, e uma terceira de assunto aleatório.

In [ ]:
frases = [
    "O carro quebrou na estrada",
    "O veículo apresentou defeito",        # Significado próximo, vocabulário diferente
    "Eu adoro pizza de calabresa"           # Assunto totalmente diferente
]

vetores = modelo.encode(frases)

# Cálculo da similaridade de cosseno da 1ª frase com as demais
sim = cosine_similarity([vetores[0]], vetores)[0]

print("=== MATRIZ DE SIMILARIDADE DE COSSENO ===")
print(f"Frase 1 vs Frase 1 (Si mesma):            {sim[0]:.4f} (Identidade)")
print(f"Frase 1 ('carro quebrou') vs Frase 2 ('veículo defeito'): {sim[1]:.4f} (Alta similaridade sem palavras em comum!)")
print(f"Frase 1 ('carro quebrou') vs Frase 3 ('pizza calabresa'): {sim[2]:.4f} (Baixa similaridade semântica)")

---
### 4. Indexação e Chunking do Documento Real (`introducaoml.pdf`)
Lemos o documento original e o particionamos em sentenças (*chunks*). Cada trecho receberá seu respectivo embedding.

In [ ]:
import pdfplumber

def ler_pdf(caminho):
    texto = ""
    with pdfplumber.open(caminho) as pdf:
        for p in pdf.pages:
            t = p.extract_text()
            if t: texto += t + " "
    return texto.replace("\n", " ").strip()

arquivo_pdf = "introducaoml.pdf"

if os.path.exists(arquivo_pdf):
    texto_natural = ler_pdf(arquivo_pdf)
else:
    texto_natural = "Machine learning é um campo da inteligência artificial que desenvolve algoritmos capazes de aprender padrões a partir de dados. Os principais tipos incluem aprendizado supervisionado e não supervisionado."

# Chunking simples por pontuação de final de sentença
chunks = [c.strip() for c in re.split(r'[.!?]', texto_natural) if len(c.strip()) > 30]

print(f"Total de chunks gerados a partir do PDF: {len(chunks)}")
print("\nExemplo de chunk indexado:")
print(f"[{chunks[0]}]")

# Vetorização de todos os pedaços do texto
emb_chunks = modelo.encode(chunks)
print(f"Matriz de embeddings: {emb_chunks.shape[0]} chunks x {emb_chunks.shape[1]} dimensões.")

---
### 5. Mini Busca Semântica (O Coração da Recuperação no RAG)
Dada uma consulta em linguagem natural:
1. Vetorizamos a pergunta.
2. Comparamos com os vetores de todos os pedaços do documento.
3. Ordenamos e retornamos os $K$ trechos mais relevantes.

In [ ]:
def buscar(pergunta, chunks, emb_chunks, top_k=2):
    emb_pergunta = modelo.encode([pergunta])
    if emb_pergunta.ndim == 1:
        emb_pergunta = emb_pergunta.reshape(1, -1)
        
    sims = cosine_similarity(emb_pergunta, emb_chunks)[0]
    
    # Índices dos chunks ordenados do mais similar para o menos similar
    melhores = sims.argsort()[::-1][:top_k]
    
    print(f"=== BUSCA: '{pergunta}' ===")
    for rank, i in enumerate(melhores, 1):
        print(f"Top {rank} [Score: {sims[i]:.4f}]: {chunks[i]}")

# Teste 1: Pergunta conceitual do domínio
buscar("O que é aprendizado de máquina?", chunks, emb_chunks, top_k=2)

print()
# Teste 2: Pergunta sobre dados rotulados
buscar("Como funciona o aprendizado supervisionado com rótulos?", chunks, emb_chunks, top_k=2)

---
### 6. Alternativa via API em Nuvem (OpenAI Embeddings)
Em vez de executar o modelo de Transformer na CPU/GPU local, sistemas corporativos frequentemente utilizam APIs gerenciadas:

```python
from openai import OpenAI
client = OpenAI() # Lê automaticamente a variável de ambiente OPENAI_API_KEY

resp = client.embeddings.create(
    model="text-embedding-3-small",
    input="Inteligência artificial é fascinante"
)

vetor_api = resp.data[0].embedding
print("Dimensões do vetor OpenAI:", len(vetor_api)) # 1536 dimensões
```

#### Modelo Local vs. API em Nuvem
| Critério | Modelo Local (`sentence-transformers`) | API em Nuvem (`OpenAI / Groq`) |
| :--- | :--- | :--- |
| **Custo** | 100% Gratuito | Pago por milhão de tokens |
| **Privacidade** | Total (dado nunca sai da sua máquina) | Requer envio dos dados a terceiros |
| **Dependência de Rede** | Roda 100% Offline | Exige conexão com a Internet e API Key |
| **Hardware** | Consome memória e processamento local | Executado na infraestrutura do provedor |


---
### Quadro-Resumo do Laboratório
- **Embedding:** Converte texto livre em um vetor matemático representativo de alta densidade.
- **Similaridade de Cosseno:** Métrica angular invariante à escala que mede a proximidade de intenção semântica.
- **Busca Semântica:** Permite recuperar informações mesmo quando o usuário utiliza sinônimos ou termos diferentes dos presentes no documento original.
